# Phase 2D — Load-flow audit

Per-province voltage and loading statistics across the three load-flow scenarios. Surfaces the calibration concerns the Phase 1 closeout flagged and the new Phase 2 findings (convergence cliff, line overloads).

**Inputs:** `load_flow_results.csv`, `buses.csv`, `bus_component_map.csv`.
**Output:** `backend/data/processed/loadflow_audit.csv` — per-(scenario, province) row with bus count, vm_pu statistics (NR only), under/over-voltage counts, and worst line loading.

In [1]:
from pathlib import Path
import pandas as pd

PROC = Path('../backend/data/processed')
results = pd.read_csv(PROC / 'load_flow_results.csv')
buses = pd.read_csv(PROC / 'buses.csv')
comp_map = pd.read_csv(PROC / 'bus_component_map.csv')
summary = pd.read_csv(PROC / 'load_flow_summary.csv')
lines = pd.read_csv(PROC / 'lines.csv')

print('Scenario summary from 2C:')
print(summary.to_string(index=False))

Scenario summary from 2C:
    scenario  factor mode  wall_s  load_mw_in_svc  gen_mw  slack_p_mw  vm_pu_min  vm_pu_max  line_load_max_pct  trafo_load_max_pct
    off_peak    0.40   nr    0.03           330.2   600.0      -232.2      0.766       1.02              179.5                81.3
morning_peak    0.85   dc    0.32           701.8   600.0       101.8        NaN        NaN              169.7                73.9
evening_peak    0.95   dc    0.33           784.3   600.0       184.3        NaN        NaN              168.6                73.1


## §1 — In-service coverage by province

Which provinces are actually in the load flow vs sitting in disconnected fragments.

In [2]:
BIG = 0
bm = comp_map.set_index('bus_id')['component_id']
buses['component_id'] = buses['bus_id'].map(bm)
buses['in_service'] = buses['component_id'] == BIG

by_prov = (
    buses.groupby('province')
    .agg(total_buses=('bus_id', 'size'),
         in_service_buses=('in_service', 'sum'),
         total_load_mw=('p_mw', lambda s: s.sum(skipna=True)),
         in_service_load_mw=('p_mw', lambda s: s[buses.loc[s.index, 'in_service']].sum(skipna=True)))
    .reset_index()
    .sort_values('total_buses', ascending=False)
)
by_prov['coverage_pct'] = (by_prov['in_service_buses'] / by_prov['total_buses'] * 100).round(1)
by_prov.round(1)

,province,total_buses,in_service_buses,total_load_mw,in_service_load_mw,coverage_pct
5,Cebu,977,421,720.0,307.1,43.1
9,Leyte,483,194,200.0,80.4,40.2
10,Negros Occidental,316,190,320.0,192.0,60.1
11,Negros Oriental,262,118,160.0,71.5,45.0
3,Bohol,251,182,120.0,86.8,72.5
8,Iloilo,158,0,250.0,0.0,0.0
13,Samar,158,46,90.0,25.4,29.1
12,Northern Samar,67,16,65.0,14.7,23.9
6,Eastern Samar,65,25,45.0,22.5,38.5
15,Southern Leyte,57,38,40.0,25.3,66.7


## §2 — Per-(scenario, province) voltage and loading audit

In [3]:
# Bus rows: scenario × bus_id with vm_pu (NR only) — join province
bus_rows = results[results['bus_id'].notna()].merge(
    buses[['bus_id', 'province', 'island', 'voltage_kv']], on='bus_id', how='left'
)

# Line rows: join voltage / submarine status via lines.csv
line_rows = results[results['line_id'].notna()].merge(
    lines[['line_id', 'voltage_kv', 'is_submarine']], on='line_id', how='left'
)

audit_rows = []
for (scen, prov), grp in bus_rows.groupby(['scenario', 'province']):
    has_vm = grp['vm_pu'].notna().any()
    audit_rows.append({
        'scenario': scen,
        'province': prov,
        'bus_count': len(grp),
        'vm_pu_min': round(grp['vm_pu'].min(), 3) if has_vm else None,
        'vm_pu_mean': round(grp['vm_pu'].mean(), 3) if has_vm else None,
        'vm_pu_max': round(grp['vm_pu'].max(), 3) if has_vm else None,
        'n_under_0p95': int((grp['vm_pu'] < 0.95).sum()) if has_vm else None,
        'n_over_1p05': int((grp['vm_pu'] > 1.05).sum()) if has_vm else None,
        'va_min_deg': round(grp['va_degree'].min(), 2),
        'va_max_deg': round(grp['va_degree'].max(), 2),
        'mode': grp['convergence_mode'].iloc[0],
    })

audit = pd.DataFrame(audit_rows).sort_values(['scenario', 'province']).reset_index(drop=True)
audit

,scenario,province,bus_count,vm_pu_min,vm_pu_mean,vm_pu_max,n_under_0p95,n_over_1p05,va_min_deg,va_max_deg,mode
0,evening_peak,Bohol,182,NaN,NaN,NaN,NaN,NaN,-13.20,-0.97,dc
1,evening_peak,Cebu,421,NaN,NaN,NaN,NaN,NaN,-12.95,3.33,dc
2,evening_peak,Eastern Samar,25,NaN,NaN,NaN,NaN,NaN,-11.69,-2.79,dc
3,evening_peak,Leyte,194,NaN,NaN,NaN,NaN,NaN,-6.13,5.60,dc
4,evening_peak,Negros Occidental,190,NaN,NaN,NaN,NaN,NaN,-18.80,-1.06,dc
5,evening_peak,Negros Oriental,118,NaN,NaN,NaN,NaN,NaN,-7.30,3.27,dc
6,evening_peak,Northern Samar,16,NaN,NaN,NaN,NaN,NaN,-9.10,-0.07,dc
7,evening_peak,Samar,46,NaN,NaN,NaN,NaN,NaN,-10.06,-0.85,dc
8,evening_peak,Southern Leyte,38,NaN,NaN,NaN,NaN,NaN,-7.61,-0.49,dc
9,morning_peak,Bohol,182,NaN,NaN,NaN,NaN,NaN,-11.08,-0.64,dc


## §3 — Headline findings

In [4]:
# 1. Provinces with zero coverage (no in-service buses → not in any scenario)
zero = by_prov[by_prov['in_service_buses'] == 0]
print(f'Provinces excluded from load flow (0 in-service buses): {len(zero)}')
for _, r in zero.iterrows():
    print(f'  {r["province"]:<25}  {int(r["total_buses"]):>4} buses, {r["total_load_mw"]:.0f} MW load not modelled')
print()

# 2. NR-scenario voltage health
nr = audit[audit['mode'] == 'nr']
print('NR scenario (off_peak) voltage health:')
worst_v = nr.nsmallest(5, 'vm_pu_min')[['province', 'bus_count', 'vm_pu_min', 'vm_pu_mean', 'n_under_0p95']]
print(worst_v.to_string(index=False))
print()

# 3. Line / transformer overloads across all scenarios
for scen in ['off_peak', 'morning_peak', 'evening_peak']:
    sl = line_rows[line_rows['scenario'] == scen]
    overloads = sl[sl['loading_percent'] > 100].sort_values('loading_percent', ascending=False)
    print(f'{scen}: {len(overloads)} overloaded edges (loading > 100%)')
    if len(overloads):
        for _, r in overloads.head(5).iterrows():
            v = r['voltage_kv'] if pd.notna(r['voltage_kv']) else '—'
            sub = 'submarine' if r.get('is_submarine') else 'overhead'
            print(f'  {r["line_id"]:<30}  {r["loading_percent"]:>6.1f}%  v={v} kV  {sub}')

Provinces excluded from load flow (0 in-service buses): 7
  Iloilo                      158 buses, 250 MW load not modelled
  Aklan                        46 buses, 75 MW load not modelled
  Antique                      25 buses, 55 MW load not modelled
  Capiz                        25 buses, 80 MW load not modelled
  Siquijor                     22 buses, 15 MW load not modelled
  Guimaras                     21 buses, 25 MW load not modelled
  Biliran                      19 buses, 22 MW load not modelled

NR scenario (off_peak) voltage health:
         province  bus_count  vm_pu_min  vm_pu_mean  n_under_0p95
Negros Occidental        190      0.766       0.901         153.0
             Cebu        421      0.823       0.940         242.0
            Bohol        182      0.883       0.960          61.0
  Negros Oriental        118      0.895       0.959          50.0
    Eastern Samar         25      0.902       0.954          13.0

off_peak: 2 overloaded edges (loading > 100%)
  l

## §4 — Save

In [5]:
out = PROC / 'loadflow_audit.csv'
audit.to_csv(out, index=False)
print(f'Wrote {out} ({len(audit)} rows)')

out_cov = PROC / 'loadflow_coverage.csv'
by_prov.round(1).to_csv(out_cov, index=False)
print(f'Wrote {out_cov} ({len(by_prov)} provinces)')

Wrote ../backend/data/processed/loadflow_audit.csv (27 rows)
Wrote ../backend/data/processed/loadflow_coverage.csv (16 provinces)
